# Languages Dataset Cleaning
This notebook cleans and prepares the languages data for further data analysis.

In [12]:
import pandas as pd

In [13]:
lan_df = pd.read_csv('../../data/languages.csv')
lan_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038762 entries, 0 to 1038761
Data columns (total 3 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   id        1038762 non-null  int64 
 1   type      1038762 non-null  object
 2   language  1038762 non-null  object
dtypes: int64(1), object(2)
memory usage: 23.8+ MB


## Dataset Overview
The Dataset contains information about movies available language options.
The data is divided in **3 columns**:
- **id**: A unique identifier for each movie.
- **type**: Indicates the type of language provided for the movie, its values can be:
    - **Language**: If the only language available is the original language of the movie
    - **Primary language**: When the movie is available in more than one language, *Primary language* type refers to the original language
    - **Spoken language**: When the movie is available in more than one language, *Spoken language* type stands to the dubbing language available
- **language**: Provides the value for each type of language.

---

## Starting the data cleaning process
The goals are:
- **Casting *type* and *language* column values to *category***
- **Handling duplicates and redundant data**: since ids have to be unique for each movie
- **Handling logical inconsistency**
- **Changing values in the *type* column**: this streamlines the dataset
- **Rename columns**: to give more unique identifiable meaning
- **Look for wrong values**: we have to be sure that the dataset does not contain outliers, wrong or unrealistic values.

### 1. Looking for null values

In [14]:
null_values = lan_df.isnull().sum()
print(null_values)

id          0
type        0
language    0
dtype: int64


There are **0 null values**, in all 3 columns.

### 2. Looking for duplicates and redundant data
   1. Checking if the tuple ('id', 'type', 'language') is duplicated

In [15]:
duplicates = lan_df.duplicated(subset=['id', 'type', 'language'], keep=False).sum()
print(duplicates)

0


There are **none** tuple *(id, type, language)* duplicates

2. Checking and handling **Inconsistent** or **Redundant Data**
The steps we take are:
   1. Grouping by *id* and *language*, calculating the number of unique values in the *type* column, creating a DataFrame with a new column named *unique_type_count* that holds the number of unique type values.
   2. Filtering only the rows which values in the *unique_type_count* column are greater than 1 and removing this last column

In [16]:
multiple_types = lan_df.groupby(['id', 'language'])['type'].nunique().reset_index(name='unique_type_count')
multiple_types = multiple_types[multiple_types['unique_type_count'] > 1][['id', 'language']]
multiple_types.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56184 entries, 3 to 982379
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        56184 non-null  int64 
 1   language  56184 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.3+ MB


There are *56184* entries.
We proceed to remove these redundant rows:
1. Merging, with a left join between the main dataframe *lan_df* and the *multiple_types* dataframe, creating a new column *_merge*
2. Keeping only the rows needed
3. Dropping the created column *_merge*

In [17]:
lan_df = lan_df.merge(multiple_types, on=['id', 'language'], how='left', indicator=True)
lan_df = lan_df[~((lan_df['type'] == 'Spoken language') & (lan_df['_merge'] == 'both'))]
lan_df = lan_df.drop(columns=['_merge'])
print(lan_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 982578 entries, 0 to 1038761
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        982578 non-null  int64 
 1   type      982578 non-null  object
 2   language  982578 non-null  object
dtypes: int64(1), object(2)
memory usage: 30.0+ MB
None


### 3. Checking and handling Logical Inconsistencies
We want to be sure that no *(id, language)* combination has more than one *Primary Language* entries

In [18]:
# Group by (id, language) and count occurrences of 'Primary language'
primary_language_count = lan_df[lan_df['type'] == 'Primary language'].groupby(['id', 'language'])['type'].count().reset_index(name='primary_language_count')

# Identify (id, language) combinations with multiple 'Primary language' entries
logical_issues = primary_language_count[primary_language_count['primary_language_count'] > 1][['id', 'language']]
print(logical_issues.info())

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        0 non-null      int64 
 1   language  0 non-null      object
dtypes: int64(1), object(1)
memory usage: 0.0+ bytes
None


There are *none*

### 4. Changing *type* column entries values
   1. Bringing all values to *lowercase*
   2. Changing *language* value to *primary language* for "standardization"
   3. Changing *spoken language* value to *dubbing language*

In [19]:
lan_df['type'] = lan_df['type'].apply(lambda x: x.lower())
lan_df['language'] = lan_df['language'].apply(lambda x: x.lower())
lan_df['type'] = lan_df['type'].replace('language', 'primary_language')
lan_df['type'] = lan_df['type'].replace('spoken language', 'dubbing language')

lan_df.head()

,id,type,language
0,1000001,primary_language,english
1,1000002,primary language,korean
2,1000002,dubbing language,english
3,1000002,dubbing language,german
5,1000003,primary language,english


In [20]:
lan_df.rename(columns={'type' : 'language_type'}, inplace=True)
lan_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 982578 entries, 0 to 1038761
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             982578 non-null  int64 
 1   language_type  982578 non-null  object
 2   language       982578 non-null  object
dtypes: int64(1), object(2)
memory usage: 30.0+ MB


### 5. **Casting *type* and *language* column values to *category***

In [21]:
lan_df['language_type'] = lan_df['language_type'].astype('category')
lan_df['language'] = lan_df['language'].astype('category')
lan_df.info()
print(lan_df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 982578 entries, 0 to 1038761
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype   
---  ------         --------------   -----   
 0   id             982578 non-null  int64   
 1   language_type  982578 non-null  category
 2   language       982578 non-null  category
dtypes: category(2), int64(1)
memory usage: 17.8 MB
        id     language_type language
0  1000001  primary_language  english
1  1000002  primary language   korean
2  1000002  dubbing language  english
3  1000002  dubbing language   german
5  1000003  primary language  english


### Saving the cleaned DataFrame

In [22]:
lan_df.to_csv('../../data/data-cleaned/languages_cleaned.csv', index=False)
print("Saved the cleaned DataFrame to CSV")

Saved the cleaned DataFrame to CSV


In [23]:
%reset -f